In [1]:
!pip install pandas tqdm requests

  Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl.metadata (19 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached numpy-2.5.2-cp314-cp314-win_amd64.whl.metadata (6.6 kB)
  Using cached tzdata-2026.3-py2.py3-none-any.whl.metadata (1.4 kB)
Using cached pandas-3.0.5-cp314-cp314-win_amd64.whl (10.0 MB)
Using cached tqdm-4.70.0-py3-none-any.whl (80 kB)
Using cached numpy-2.5.2-cp314-cp314-win_amd64.whl (12.6 MB)
Using cached tzdata-2026.3-py2.py3-none-any.whl (348 kB)

   ---------------------------------------- 0/4 [tzdata]
   ---------------------------------------- 0/4 [tzdata]
   ---------- ----------------------------- 1/4 [tqdm]
   ---------- ----------------------------- 1/4 [tqdm]
   -------------------- ------------------- 2/4 [numpy]
   -------------------- ------------------- 2/4 [numpy]
   -------------------- ------------------- 2/4 [numpy]
   -------------------- ------------------- 2/4 [numpy]
   -------------------- ------------------- 2/4 [n


[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:

import os
import time
import requests
import pandas as pd
from pathlib import Path
from tqdm import tqdm

# =====================================================
# Configuration
# =====================================================

OUTPUT_DIR = Path("AI_Knowledge_Base")
OUTPUT_DIR.mkdir(exist_ok=True)

HEADERS = {
    "User-Agent": "AIML-RAG-Dataset-Builder/1.0"
}

SEARCH_LIMIT = 10
MAX_RETRIES = 3
SLEEP_TIME = 1

# =====================================================
# Seed Keywords
# =====================================================

KEYWORDS = [

    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Large Language Model",
    "Natural Language Processing",
    "Data Science",
    "Computer Vision",
    "Generative AI",
    "Transformer",
    "Neural Network",
    "Prompt Engineering",
    "Retrieval Augmented Generation",
    "Semantic Search",
    "Vector Database",
    "Feature Engineering"

]

# =====================================================
# Search Wikipedia
# =====================================================

def search_articles(keyword, limit=10):

    url = "https://en.wikipedia.org/w/api.php"

    params = {
        "action": "query",
        "list": "search",
        "srsearch": keyword,
        "srlimit": limit,
        "format": "json"
    }

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    titles = []

    for article in data["query"]["search"]:
        titles.append(article["title"])

    return titles

# =====================================================
# Download Wikipedia Article
# =====================================================

def download_article(title):

    url = "https://en.wikipedia.org/w/api.php"

    params = {

        "action": "query",
        "format": "json",
        "titles": title,
        "redirects": 1,
        "prop": "extracts|info",
        "inprop": "url",
        "explaintext": 1

    }

    response = requests.get(
        url,
        params=params,
        headers=HEADERS,
        timeout=30
    )


    for retry in range(MAX_RETRIES):
        try:
            response = requests.get(
                url,
                params=params,
                headers=HEADERS,
                timeout=30
            )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After", "5")
                wait_time = int(retry_after) + retry
                print(f"Rate limited. Waiting {wait_time} seconds...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()

            data = response.json()
            pages = data.get("query", {}).get("pages", {})

            page = list(pages.values())[0]

            if "missing" in page:
                return None

            text = page.get("extract", "").strip()

            if not text:
                return None
            

            return {
                "title": page["title"],
                "text": text,
                "url": page.get(
                    "fullurl",
                    f"https://en.wikipedia.org/wiki/{page['title'].replace(' ', '_')}"
                )
            }

        except Exception as e:
            if retry == MAX_RETRIES - 1:
                raise e

            time.sleep(SLEEP_TIME * (retry + 1))

    return None

        

# =====================================================
# Safe filename
# =====================================================

def clean_filename(name):

    invalid = '<>:"/\\|?*'

    for ch in invalid:
        name = name.replace(ch, "_")

    return name

# =====================================================
# Main Download Loop
# =====================================================

metadata = []
errors = []

visited = set()

print("="*60)
print("Searching Wikipedia")
print("="*60)

candidate_titles = []

for keyword in KEYWORDS:

    print(f"\nSearching : {keyword}")

    try:

        results = search_articles(keyword, SEARCH_LIMIT)

        candidate_titles.extend(results)

        print(f"Found {len(results)} articles")

    except Exception as e:

        print(e)

candidate_titles = sorted(set(candidate_titles))

print("\n")
print("="*60)
print(f"Total Unique Articles Found : {len(candidate_titles)}")
print("="*60)

# =====================================================
# Download
# =====================================================

for title in tqdm(candidate_titles):

    if title in visited:
        continue

    visited.add(title)

    success = False

    for retry in range(MAX_RETRIES):

        try:

            page = download_article(title)

            if page is None:
                raise Exception("Page not found")

            filename = clean_filename(page["title"]) + ".txt"

            filepath = OUTPUT_DIR / filename

            if filepath.exists():

                success = True

                break

            with open(
                filepath,
                "w",
                encoding="utf-8"
            ) as f:

                f.write(page["text"])

            metadata.append({

                "Title": page["title"],
                "URL": page["url"],
                "Words": len(page["text"].split()),
                "Characters": len(page["text"]),
                "File": filename

            })

            success = True

            break

        except Exception as e:

            if retry == MAX_RETRIES - 1:

                errors.append({

                    "Title": title,
                    "Error": str(e)

                })

            time.sleep(SLEEP_TIME)

# =====================================================
# Save CSV
# =====================================================

metadata_df = pd.DataFrame(metadata)

metadata_df.to_csv(
    OUTPUT_DIR / "metadata.csv",
    index=False
)

errors_df = pd.DataFrame(errors)

errors_df.to_csv(
    OUTPUT_DIR / "error_log.csv",
    index=False
)

print("\n")
print("="*60)
print("Download Completed")
print("="*60)
print(f"Articles Downloaded : {len(metadata_df)}")
print(f"Failed Downloads    : {len(errors_df)}")
print(f"Folder              : {OUTPUT_DIR}")
print("="*60)

Searching Wikipedia

Searching : Artificial Intelligence
Found 10 articles

Searching : Machine Learning
Found 10 articles

Searching : Deep Learning
Found 10 articles

Searching : Large Language Model
Found 10 articles

Searching : Natural Language Processing
Found 10 articles

Searching : Data Science
Found 10 articles

Searching : Computer Vision
Found 10 articles

Searching : Generative AI
Found 10 articles

Searching : Transformer
Found 10 articles

Searching : Neural Network
Found 10 articles

Searching : Prompt Engineering
429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Prompt+Engineering&srlimit=10&format=json

Searching : Retrieval Augmented Generation
429 Client Error: Too Many Requests for url: https://en.wikipedia.org/w/api.php?action=query&list=search&srsearch=Retrieval+Augmented+Generation&srlimit=10&format=json

Searching : Semantic Search
429 Client Error: Too Many Requests for url: https://en.wikipedia.o

  0%|          | 0/89 [00:00<?, ?it/s]

Rate limited. Waiting 37 seconds...


  6%|▌         | 5/89 [00:47<07:11,  5.14s/it]

Rate limited. Waiting 50 seconds...


...

Rate limited. Waiting 49 seconds...


 96%|█████████▌| 85/89 [16:48<00:25,  6.43s/it]

Rate limited. Waiting 48 seconds...


100%|██████████| 89/89 [17:45<00:00, 11.98s/it]



Download Completed
Articles Downloaded : 42
Failed Downloads    : 0
Folder              : AI_Knowledge_Base


In [3]:
import pandas as pd

errors_df = pd.read_csv(OUTPUT_DIR / "error_log.csv")

print(f"Total errors: {len(errors_df)}")
print("\nFirst 10 errors:")

errors_df.head(10)

Total errors: 43

First 10 errors:


,Title,Error
0,A.I. Artificial Intelligence,429 Client Error: Too Many Requests for url: h...
1,AI Overviews,429 Client Error: Too Many Requests for url: h...
2,AI boom,429 Client Error: Too Many Requests for url: h...
3,AI slop,429 Client Error: Too Many Requests for url: h...
4,Active learning (machine learning),429 Client Error: Too Many Requests for url: h...
5,Adversarial machine learning,429 Client Error: Too Many Requests for url: h...
6,AlexNet,429 Client Error: Too Many Requests for url: h...
7,Computer vision syndrome,429 Client Error: Too Many Requests for url: h...
8,Conference on Computer Vision and Pattern Reco...,429 Client Error: Too Many Requests for url: h...
9,Controlled natural language,429 Client Error: Too Many Requests for url: h...
